In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

In [10]:
def fractional_release(t, D, R, n_terms=50):
    """
    Compute M_t / M_inf for diffusion out of a sphere.

    Parameters
    ----------
    t : float or array_like
        Time(s) at which to evaluate the release. Same time units as implied by D.
    D : float
        Diffusion coefficient (e.g. m^2/s). Must be > 0.
    R : float
        Sphere radius (same length units as D, e.g. m). Must be > 0.
    n_terms : int, optional
        Number of terms in the series (default 50 — gives machine precision
        for any t > 0 because of the exponential decay in n^2).

    Returns
    -------
    Mt_over_Minf : ndarray
        Fractional mass released, in [0, 1]. Same shape as `t`.
    """
    if D <= 0 or R <= 0:
        raise ValueError("D and R must be positive.")
    if n_terms < 1:
        raise ValueError("n_terms must be >= 1.")

    t = np.asarray(t, dtype=float)

    # n = 1, 2, ..., n_terms  -> shape (n_terms,)
    n = np.arange(1, n_terms + 1)

    # Broadcast: t has shape (...,), n has shape (n_terms,)
    # exponent has shape (..., n_terms)
    exponent = -D * (n**2) * (np.pi**2) * t[..., np.newaxis] / (R**2)
    series = np.sum(np.exp(exponent) / n**2, axis=-1)

    result = 1.0 - (6.0 / np.pi**2) * series

    # At t = 0 the series sums to pi^2/6 exactly, giving 0.0 — but clip tiny
    # negative values from floating-point error.
    return np.clip(result, 0.0, 1.0)

In [11]:
df_profiles = pd.read_excel("plga_dataset/mp_dataset_processed.xlsx")
df_summary = pd.read_excel("plga_dataset/crank_fit.xlsx")

In [ ]:
df_release_new = pd.DataFrame()
df_release_new_even = pd.DataFrame()

number_of_samples = 50 # how many points in uniform release
indices = np.linspace(0, 1, number_of_samples) # linear indics of points in uniform release

for i in range(len(df_profiles['Formulation Index'].unique())):

    selected_index = i + 1
    df_profile_0 = df_profiles[df_profiles['Formulation Index'] == selected_index].reset_index(drop=True)
    df_profile_0 = df_profile_0[df_profile_0['Release'] >= 0].reset_index(drop=True)
    df_profile_0 = df_profile_0[df_profile_0['Release'] <= 1].reset_index(drop=True)

    df_summary_0 = df_summary[df_summary['Formulation Index'] == selected_index].reset_index(drop=True)
    
    particle_radius = df_summary_0['Particle Radius'].iloc[0] * 10**-6 # micrometers
    D = df_summary_0['Crank_D'].iloc[0] # m^2/sec
    time_array = df_profile_0['Time'] * 24 * 3600 # sec

    diffusion_release = fractional_release(time_array, D, particle_radius, n_terms=170)

    df_profile_0['Crank_D'] = D
    df_profile_0['Crank_Release'] = diffusion_release
    
    df_release_new = pd.concat([df_release_new, df_profile_0])



    ## START - Construct new df with same time samplign
    max_release_time = time_array.max() # seconds
    # even_release_time_array = max_release_time * (np.exp(indices) - 1) / (np.exp(1) - 1) # Exponential distribution
    even_release_time_array = max_release_time * (indices ** 1.5) # Power law distribution

    even_release_fraction_array = fractional_release(even_release_time_array, D, particle_radius, n_terms=170)

    df_profile_0_even = pd.DataFrame({
        "Time" : even_release_time_array / (24*3600),
        "Crank_Release" : even_release_fraction_array
    })

    df_profile_0_even['Crank_D'] = D

    for col in df_profile_0.drop(columns=['Time', 'Release', 'Crank_D', 'Crank_Release']).columns:
        df_profile_0_even[col] = df_profile_0[col].iloc[0]

    df_release_new_even = pd.concat([df_release_new_even, df_profile_0_even])
    

    ## END - Construct new df with same time samplign
    
    plt.plot(df_profile_0['Time'], df_profile_0['Release'], marker='x', label='Original')
    plt.plot(df_profile_0['Time'], diffusion_release, marker='>', label='Crank')
    plt.plot(even_release_time_array / 24 / 3600 , even_release_fraction_array, marker='x', label='Crank even')
    plt.legend()
    plt.grid()
    plt.show()
    
    # break

df_release_new.reset_index(drop=True)
df_release_new_even.reset_index(drop=True)

In [13]:
## Save dataframe with 50 time points per particle
df_release_new_even.to_excel("plga_dataset/release_dataset_with_Crank_release_even.xlsx", index=False)